In [0]:
drop table if exists globalretail.gr_silver.transaction;
create table if not exists globalretail.gr_silver.transaction
as
select
 CAST(transaction_id AS STRING) AS transaction_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(product_id AS INT) AS product_id,
    CAST(quantity AS INT) AS quantity,
    CAST(total_amount AS DOUBLE) AS total_amount,
    CAST(transaction_date AS STRING) AS transaction_date,
    CAST(payment_method AS STRING) AS payment_method,
    CAST(store_type AS STRING) AS store_type,
    current_timestamp() AS ingestion_date
from globalretail.gr_bronze.transaction

In [0]:
DESCRIBE TABLE globalretail.gr_bronze.transaction;

In [0]:
select * from globalretail.gr_silver.transaction

- Quantity and total_amount set to 0 if negative
- date casting to ensure consistent date format
- customer id , product id no null value
- order status based on quantity and total amount
- 

In [0]:
MERGE INTO globalretail.gr_silver.transaction AS target
USING globalretail.gr_bronze.transaction AS source
ON target.transaction_id = source.transaction_id


-- ✅ Update existing records
WHEN MATCHED THEN UPDATE SET
target.transaction_id = source.transaction_id,
target.customer_id = source.customer_id,
target.product_id = source.product_id,
target.quantity = source.quantity,
target.total_amount = source.total_amount,
target.transaction_date = source.transaction_date,
target.payment_method = source.payment_method,
target.store_type = source.store_type,
target.ingestion_date = current_timestamp()

-- ✅ Insert new records
WHEN NOT MATCHED THEN INSERT (
transaction_id,
customer_id,
product_id,
quantity,
total_amount,
transaction_date,
payment_method,
store_type
)
VALUES (
    source.transaction_id,
    source.customer_id,
    source.product_id,
    source.quantity,
    source.total_amount,
    source.transaction_date,
    source.payment_method,
    source.store_type
);
select * from globalretail.gr_bronze.transaction


In [0]:
CREATE OR REPLACE TABLE globalretail.gr_silver.transaction_transform
USING DELTA
AS
SELECT
 transaction_id,
 customer_id,
 product_id,
 case 
 when quantity < 0 then 0
 else quantity
 end as quantity,
  total_amount,
    CAST(transaction_date AS DATE) AS transaction_date,
  payment_method,
  store_type,
    current_timestamp() AS ingestion_date
from globalretail.gr_bronze.transaction
WHERE transaction_id is not NULL
and customer_id is not NULL
and product_id is not NULL

In [0]:
select * from globalretail.gr_silver.transaction_transform